# Quick start

This notebook converts the IEEE 13-node feeder from OpenDSS to CIM and back. It runs from `docs/guide/` in the
repository, which holds the deck at `../../tests/decks/IEEE/IEEE13_CDPSM/IEEE13_CDPSM.dss`. Install the package first
([Installation](installation.md)).

| Step | Call | Touches files |
|---|---|---|
| Read | `DssReader(path).read_all()` returns a `DssModel` | reads the deck and its `Redirect`s |
| Import | `dss_to_cim(model, name=...)` returns a cim18v17 network | writes `.cimtbl` artifacts to a temporary directory unless you pass `artifact_dir=` |
| Export | `DssExporter().to_model(network)` returns a `DssModel` | none |
| Write | `DssWriter().write(model, path)` | writes the deck |

Only the reader and the writer touch the filesystem.

## Read the deck

`DssReader` follows `Redirect`s and `~` continuation lines, evaluates RPN values, and applies `Edit`s and `Open`/`Close`
as the engine would. `read_all` returns a `DssModel`: records keyed by their DSS class.

In [1]:
from pathlib import Path

from cimhub_opendss import DssReader

deck = Path("../../tests/decks/IEEE/IEEE13_CDPSM/IEEE13_CDPSM.dss")
model = DssReader(deck).read_all()
for dss_class, records in model.graph.items():
    print(f"{dss_class.__name__:10} {len(records)}")

Vsource    1
Transformer 6
Regcontrol 3
Linecode   7
Load       16
Capacitor  2
Line       16
Relay      1
Fuse       1
Recloser   1
Pvsystem   2
Storage    3
Xfmrcode   1
Options    1
Buscoords  22


Each record is a dataclass from the LinkML schema. Slots keep the units the deck used: this load's `kW` is in kW and its
`kV` in kV.

In [2]:
from cimhub_opendss.schema import opendss_schema as schema

print(model.list_by_class(schema.Load)[0])

Load({
  'name': '611',
  'bus1': '611.3',
  'phases': 1,
  'kV': 2.4,
  'kW': 170.0,
  'model': LoadModel(text='ConstantI', description='5: Constant Current Magnitude', meaning='https://gridappsd.github.io/dss-graph/LoadModel.5'),
  'conn': ConnectionType(text='wye', description='Wye (star) connection', meaning='http://iec.ch/TC57/CIM100#WindingConnection.Y'),
  'kvar': 80.0
})


## Import to CIM

`dss_to_cim` maps the records to `.cimtbl` text and has cimtables build the CIM network from it. `name` is the master
document's `Model` and file stem. `artifact_dir` keeps the text; here it goes to a temporary directory.

In [3]:
import tempfile

from cimhub_opendss import dss_to_cim

work = Path(tempfile.mkdtemp())
network = dss_to_cim(model, name="IEEE13_CDPSM", artifact_dir=work / "cimtbl")

The network is a cim-graph model. `network.graph` holds its objects by CIM class. The deck's 16 `Line`s are 11
`ACLineSegment`s and five switches, and its six `Transformer`s are three `PowerTransformer`s and three
`VoltageRegulator`s (the regulated, equal-voltage ones).

In [4]:
import cimprofiles.cim18.cim18v17 as cim

for cim_class in (cim.ConnectivityNode, cim.ACLineSegment, cim.LoadBreakSwitch, cim.Fuse, cim.Recloser,
                  cim.PowerTransformer, cim.VoltageRegulator, cim.EnergyConsumer, cim.LinearShuntCompensator,
                  cim.PowerElectronicsConnection):
    print(f"{cim_class.__name__:28} {len(network.graph.get(cim_class, {}))}")

ConnectivityNode             22
ACLineSegment                11
LoadBreakSwitch              3
Fuse                         1
Recloser                     1
PowerTransformer             3
VoltageRegulator             3
EnergyConsumer               16
LinearShuntCompensator       2
PowerElectronicsConnection   5


To read a value, use a [CIMHub core getter](https://pnnl-cim-tools.github.io/CIMHub/core/guide/getters/). It converts
CIM's SI value to the unit you name. Load 611 is a 170 kW, 80 kVAr load on phase C.

In [5]:
from cimhub_core.getters.load import get_phase_loads

load = next(obj for obj in network.graph[cim.EnergyConsumer].values() if obj.name == "611")
loads = get_phase_loads(load, "kW", "kVAr")
print(loads.total)
print([phase.name for phase in loads.phases])

PhasePower(p=170.0, q=80.0)
['C']


## The `.cimtbl` artifacts

`artifact_dir` holds one file per DSS table plus the master, which `Import`s the others in dependency order (catalogs
before the equipment that uses them).

In [6]:
artifacts = work / "cimtbl"
print(sorted(path.name for path in artifacts.iterdir()))
print()
imports = [line for line in (artifacts / "IEEE13_CDPSM.cimtbl").read_text().splitlines() if line.startswith("Import")]
print("\n".join(imports))

['IEEE13_CDPSM.cimtbl', 'bus.cimtbl', 'buscoords.cimtbl', 'capacitor.cimtbl', 'line.cimtbl', 'linecode.cimtbl', 'linegeometry.cimtbl', 'load.cimtbl', 'pvsystem.cimtbl', 'regcontrol.cimtbl', 'storage.cimtbl', 'transformer.cimtbl', 'vsource.cimtbl', 'xfmrcode.cimtbl', 'zip_fractions.cimtbl']

Import bus.cimtbl
Import zip_fractions.cimtbl
Import linegeometry.cimtbl
Import linecode.cimtbl
Import xfmrcode.cimtbl
Import vsource.cimtbl
Import line.cimtbl
Import transformer.cimtbl
Import regcontrol.cimtbl
Import load.cimtbl
Import capacitor.cimtbl
Import pvsystem.cimtbl
Import storage.cimtbl
Import buscoords.cimtbl


Values are copied as the deck gave them, and the unit is in the column header (`p (kW)`). A comment says what the
importer defaulted.

In [7]:
print("\n".join((artifacts / "load.cimtbl").read_text().splitlines()[6:14]))

Profile = cim18v17

Table EnergyConsumer: mRID, name, node, phases, phaseConnection, grounded, p (kW), q (kVAr), customerCount, LoadResponse, normallyInService, BaseVoltage, EquipmentContainer=fdr_ieee13nodeckt
    # defaulted: NumCust=1, enabled=true
    ld_611,   611,   611,   C,    Y, true,  170.0,  80.0,               1, zip_constant_i,  true, bv_4p16kv
    # defaulted: NumCust=1, enabled=true
    ld_634a,  634a,  634,   A,    Y, true,  160.0,  110.0,              1, zip_constant_pq, true, bv_480v
    # defaulted: NumCust=1, enabled=true


The master's comment lists every DSS class in the deck that no mapper reads. IEEE 13 has one `Relay`, so it is counted
and not imported.

In [8]:
print("\n".join((artifacts / "IEEE13_CDPSM.cimtbl").read_text().splitlines()[:3]))

# OpenDSS deck IEEE13_CDPSM
# Mapping: cimhub_opendss/docs/CIMTBL_MAPPING.md#master
# not mapped: Relay 1


## Export to a deck

`DssExporter().to_model` reads the CIM network through the core getters and returns a `DssModel`, the same shape the reader
returns. `name_source="name"` writes each DSS name from the CIM `name`; the default is the `mRID`.

In [9]:
from cimhub_opendss import DssExporter

exported = DssExporter().to_model(network, name_source="name")
for dss_class, records in exported.graph.items():
    print(f"{dss_class.__name__:10} {len(records)}")

Vsource    1
Linecode   7
Line       16
Fuse       1
Recloser   1
Xfmrcode   1
Transformer 6
Regcontrol 3
Load       16
Capacitor  2
Pvsystem   2
Storage    3
Buscoords  22
Options    1


`DssWriter().write` creates the deck and its folders. A deck with bus coordinates also gets a CSV beside it.

In [10]:
from cimhub_opendss import DssWriter

written = work / "out" / "IEEE13_CDPSM.dss"
DssWriter().write(exported, written)
print(sorted(path.name for path in written.parent.iterdir()))
for line in written.read_text().splitlines()[:3]:
    print(line[:110])
print([line[:100] for line in written.read_text().splitlines() if line.startswith("new Load.611")])

['IEEE13_CDPSM.dss', 'IEEE13_CDPSM_buscoords.csv']
Clear
Set defaultbasefreq=60
New Circuit.IEEE13Nodeckt phases=3 bus1=sourcebus.1.2.3 basekv=115 pu=1 angle=30 R1=0.160376682055275 X1=0.641
['new Load.611 phases=1 bus1=611.3 kV=2.40177711982884 kW=170 model=5 conn=wye kvar=80 NumCust=1 enabl']


The written deck reads back through `DssReader`, with the same counts:

In [11]:
again = DssReader(written).read_all()
print(again.count(schema.Line), again.count(schema.Load), again.count(schema.Transformer))

16 16 6


`cim_to_dss(network, path)` does the export and the write in one call, and `dss_to_cim` does the read-side
mapping of an already-read model. Chained:

```python
from cimhub_opendss import DssReader, cim_to_dss, dss_to_cim

network = dss_to_cim(DssReader(deck).read_all(), name="IEEE13_CDPSM")
cim_to_dss(network, "out/IEEE13_CDPSM.dss")
```

## Next

- [Import](import.md): options, the artifact files and what raises.
- [Export](export.md): names, the circuit source, switch state and what raises.
- [Round trip](roundtrip.md): checking a converted deck against the original in the engine.
- [What converts](coverage.md): the DSS classes that import and export.